# Filtro 1

## Entreno de modelo

### Importación de librerías

In [ ]:
import os
import numpy as np
import joblib
from deepface import DeepFace
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report
from tqdm import tqdm
import kagglehub

### Importación de dataset y definición de categorías

In [2]:
DATASET_ID  = "saramhai/people-with-and-without-glasses-dataset"
DOWNLOAD_PATH  = kagglehub.dataset_download(DATASET_ID)

CLASS_LABELS  = ["glasses", "no_glasses"] 
FACE_MODEL = "ArcFace"
MODEL_FILENAME = "glasses_classifier.pkl"
LABELS_FILE = "glasses_labels.pkl"

embeddings = []
labels = []


### Entrenamiento del dataset

In [ ]:
base_path = os.path.join(DOWNLOAD_PATH , "images")

for label_id, label_name in enumerate(CLASS_LABELS):
    label_path = os.path.join(base_path, label_name)
    if not os.path.exists(label_path):
        print(f"¡Aviso! No se encontró la carpeta: {label_path}")
        continue

    img_files = [f for f in os.listdir(label_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    # Mezclar aleatoriamente
    np.random.shuffle(img_files)

    print(f"\nProcesando categoría: {label_name} (Usando {len(img_files)} imágenes)")

    # Extraer embeddings
    for filename in tqdm(selected_files, desc=label_name):
        file_path = os.path.join(label_path, filename)
        try:
            rep_data = DeepFace.represent(
                img_path=file_path,
                model_name=FACE_MODEL,
                enforce_detection=False,
                detector_backend='skip'
            )

            face_vector = rep_data[0]["embedding"]
            embeddings.append(face_vector)
            labels.append(label_id)

        except Exception as err:
            if 'face could not be detected' not in str(err):
                print(f"Error procesando {file_path}: {err}")

if not embeddings:
    print("Error: No se extrajo ningún embedding. Verifica el dataset.")
    exit()

embeddings = np.array(embeddings)
labels = np.array(labels)

print(f"\nTotal de embeddings extraídos: {embeddings.shape[0]}")
print(f"Dimensiones de cada embedding: {embeddings.shape[1]}")
print(f"Distribución de clases: {np.bincount(labels)}")

# Evaluación con train/test split
print("\nEvaluando modelo con split 70/30...")
X_train, X_test, y_train, y_test = train_test_split(
    embeddings, labels, test_size=0.3, random_state=42, stratify=labels
)

temp_model = SVC(kernel='linear', probability=True, random_state=42)
temp_model.fit(X_train, y_train)

y_pred = temp_model.predict(X_test)
print("\n" + classification_report(y_test, y_pred, target_names=CLASS_LABELS))

# Entrenamiento final con todos los datos
print("\nEntrenando modelo final con todos los datos...")
best_model = SVC(kernel='linear', probability=True, random_state=42)
best_model.fit(embeddings, labels)

# Guardar modelo y categorías
joblib.dump(best_model, MODEL_FILENAME)
joblib.dump(CLASS_LABELS, LABELS_FILE)

print(f"\nModelo: '{MODEL_FILENAME}'")
print(f"Categorías: '{LABELS_FILE}'")